# Boston Housing Price Prediction

Neural network regression on the classic Boston Housing dataset using TensorFlow / Keras. Predicts median house price from `RM`, `LSTAT`, and `PTRATIO`. Trained model + scaler are persisted to `models/` so the website can serve a live GUI.

## Part 0 — Setup

### 0.1 Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.callbacks import EarlyStopping

### 0.2 Early stopping callback

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

## Part 1 — Load and Inspect the Dataset

### 1.1 Load Boston housing CSV

In [ ]:
df = pd.read_csv('datasets/Boston1.csv')
df.rename(columns={'medv': 'price'}, inplace=True)
df.head()

### 1.2 Dataset info & null check

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

**Q1.** Rows × Columns → `506 × 14`  
**Q2.** Columns with nulls → `0`

## Part 2 — Feature Selection (Correlation)

### 2.1 Correlation matrix

In [ ]:
corr = df.corr()
corr

### 2.2 Heatmap

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', annot=True, fmt='.2f')
plt.title('Feature Correlation Heatmap')
plt.show()

### 2.3 Sort correlation with price

In [ ]:
corr['price'].sort_values(ascending=False)

**Top features by |correlation|:** `lstat`, `ptratio`, `rm`.

## Part 3 — Prepare the Data

### 3.1 Features and target

In [ ]:
X = df[['rm', 'lstat', 'ptratio']]
y = df['price']

### 3.2 Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)
print('Train:', X_train.shape, '  Test:', X_test.shape)

### 3.3 Feature scaling

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

## Part 4 — Build a Neural Network

### 4.1 Define model architecture

In [ ]:
model = Sequential([
    Input(shape=(X_train_s.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])
model.summary()

### 4.2 Compile

In [ ]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

## Part 5 — Train the Model

### 5.1 Train 100 epochs

In [ ]:
history_100 = model.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    verbose=0,
    callbacks=[early_stop]
)
print('Final train loss:', history_100.history['loss'][-1])
print('Final val loss:  ', history_100.history['val_loss'][-1])

### 5.2 Continue training for 300 epochs (fresh model)

In [ ]:
model_300 = Sequential([
    Input(shape=(X_train_s.shape[1],)),
    Dense(32, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1)
])
model_300.compile(optimizer='adam', loss='mse', metrics=['mae'])

history_300 = model_300.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=300,
    batch_size=16,
    verbose=0,
    callbacks=[early_stop]
)
print('Final train loss:', history_300.history['loss'][-1])
print('Final val loss:  ', history_300.history['val_loss'][-1])

### 5.3 Plot training vs validation loss

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history_100.history['loss'], label='train')
ax[0].plot(history_100.history['val_loss'], label='val')
ax[0].set_title('100 epochs'); ax[0].set_xlabel('Epoch'); ax[0].set_ylabel('Loss'); ax[0].legend()
ax[1].plot(history_300.history['loss'], label='train')
ax[1].plot(history_300.history['val_loss'], label='val')
ax[1].set_title('300 epochs'); ax[1].set_xlabel('Epoch'); ax[1].set_ylabel('Loss'); ax[1].legend()
plt.tight_layout(); plt.show()

## Part 6 — Model Evaluation

### 6.1 Predict on test set

In [ ]:
y_pred_100 = model.predict(X_test_s, verbose=0)
y_pred_300 = model_300.predict(X_test_s, verbose=0)

### 6.2 Compute metrics

In [ ]:
def metrics(y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2   = r2_score(y_true, y_pred)
    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}

results = pd.DataFrame({
    '100 epochs': metrics(y_test, y_pred_100),
    '300 epochs': metrics(y_test, y_pred_300),
}).T
results

## Part 7 — Single Prediction

### 7.1 Sample house features (RM, LSTAT, PTRATIO)

In [ ]:
sample = pd.DataFrame([[6.0, 12.0, 15.0]], columns=['rm', 'lstat', 'ptratio'])
sample

### 7.2 Predict price

In [ ]:
sample_scaled  = scaler.transform(sample)
predicted      = model_300.predict(sample_scaled, verbose=0)[0][0]
predicted_usd  = predicted * 1000  # MEDV is in $1000s
print(f'Predicted (raw MEDV): {predicted:.2f}')
print(f'Predicted price (USD): ${predicted_usd:,.2f}')

### 7.3 Multiple sample predictions

In [ ]:
samples = pd.DataFrame(
    [
        [6.0, 12.0, 15.0],
        [12.7, 6.5, 53.5],
        [4.2, 67.5, 19.5],
    ],
    columns=['rm', 'lstat', 'ptratio'],
)
samples_scaled = scaler.transform(samples)
preds          = model_300.predict(samples_scaled, verbose=0).flatten()
samples['predicted_price_usd'] = (preds * 1000).round(2)
samples

## Part 8 — Persist Model for Web GUI

Save the trained model + scaler so the website backend can load them and serve live predictions through the in-page form (no Anvil).

In [ ]:
MODELS_DIR = Path('models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

model_300.save(MODELS_DIR / 'boston_house_price.keras')
joblib.dump(scaler, MODELS_DIR / 'boston_house_price_scaler.pkl')

print('Saved:')
print(' -', MODELS_DIR / 'boston_house_price.keras')
print(' -', MODELS_DIR / 'boston_house_price_scaler.pkl')

## Part 9 — Prediction Helper

Reusable function the website backend calls.

In [ ]:
def predict_price(rm: float, lstat: float, ptratio: float) -> float:
    """Predict house price in USD given RM, LSTAT, PTRATIO."""
    sample        = np.array([[rm, lstat, ptratio]])
    sample_scaled = scaler.transform(sample)
    pred          = model_300.predict(sample_scaled, verbose=0)[0][0]
    return float(pred * 1000)

predict_price(6.0, 12.0, 15.0)